# Gobernanza y Seguridad de Datos - Catalogo `datos_empresas`

Politicas de seguridad implementadas sobre las tablas reales del proyecto, siguiendo los pilares de gobernanza vistos en clase: **permisos (RBAC)**, **enmascarado de columnas (PII)**, **filtros de fila (row filters)** y **auditoria**.

## Por que estas columnas y no otras

A diferencia del ejemplo del profe (que trabaja con victimas de incidentes viales, con PII evidente como direccion y edad), nuestro proyecto **ya aplica minimizacion de datos desde el notebook `03_Silver_RUES`**: no se incluyen en Silver ni en Gold los nombres del representante legal ni los nombres/apellidos de personas naturales -esa informacion vive solo en Bronze, con acceso mas restringido.

Lo que SI queda en Gold y requiere gobernanza son **cuasi-identificadores**: columnas que, combinadas, podrian re-identificar a una persona o empresa aunque no sean datos de contacto directos.

**Columnas sensibles en `gold.de_dim_empresas`:**
* `numero_identificacion_normalizado` / `nit_normalizado` — documento de identidad o NIT (cuasi-identificador fuerte).
* `razon_social` — nombre de la empresa (identifica directamente a personas naturales que registran su actividad bajo su propio nombre).
* `camara_comercio` — usada tambien como criterio de **row filter** por region.

**Columnas sensibles en `gold.de_fact_contratos`:**
* `documento_proveedor_normalizado` — documento del proveedor (persona natural o juridica).
* `proveedor_adjudicado` — nombre del proveedor.

---
## Paso 0: Explorar el entorno de gobernanza actual (solo lectura)

In [ ]:
%sql
SHOW GRANTS ON CATALOG datos_empresas;

**Resultado real:** 0 filas. Hoy nadie tiene un `GRANT` explicito sobre el catalogo -solo el `OWNER` tiene acceso implicito. Es exactamente el escenario que un data steward deberia corregir: sin RBAC explicito, cualquiera con acceso al workspace que sea `OWNER` o admin ve todo sin restriccion. El Laboratorio 1 corrige esto.

---
## Laboratorio 1: Permisos por rol (RBAC) con GRANT/REVOKE

**Roles reales del ejercicio** (los mismos 3 grupos que se usan en el Laboratorio 2b, creados de verdad en el Account Console -no son un ejemplo teorico):

| Rol | Grupo en Databricks | Acceso |
|---|---|---|
| Admin (dueño del catalogo) | `admin` | `ALL PRIVILEGES` sobre todo el catalogo -mantiene el pipeline y ve todo sin mascara |
| Recursos Humanos | `grupo_recursos_humanos` | `USE CATALOG`, `USE SCHEMA`, `SELECT` sobre `gold` -el acceso base; las columnas de valores quedan igual ocultas por el column mask del Laboratorio 2b |
| Finanzas | `grupo_finanzas` | `USE CATALOG`, `USE SCHEMA`, `SELECT` sobre `gold` -mismo acceso base; el documento del proveedor queda oculto por el mismo mecanismo |

**Nota de diseno:** el RBAC (este laboratorio) y el column mask (Laboratorio 2b) trabajan en capas distintas. El `GRANT` decide *si* un grupo puede consultar la tabla; el column mask decide *que columnas* ve dentro de esa consulta. RRHH y Finanzas tienen el mismo `GRANT` (ambos pueden `SELECT` sobre `gold`), pero ven columnas diferentes gracias a la mascara -el permiso de tabla no alcanza para expresar esa diferencia, por eso se necesitan los dos mecanismos juntos.

(Los grupos `grupo_datos_empresas_ingenieria` y `grupo_datos_empresas_analistas` de una version anterior de este notebook no se van a crear -se reemplazaron por estos 3 roles, que si se probaron en vivo con compañeros reales en el Laboratorio 2b.)

In [ ]:
%sql
-- Grupos reales, creados en el Account Console (ver Laboratorio 2b para los pasos).
-- Confirmado en vivo: is_account_group_member('admin') ya da TRUE para esta cuenta.

-- Admin: control total sobre todas las capas
GRANT ALL PRIVILEGES ON CATALOG datos_empresas TO `admin`;

-- Recursos Humanos: solo lectura, solo sobre Gold
GRANT USE CATALOG ON CATALOG datos_empresas TO `grupo_recursos_humanos`;
GRANT USE SCHEMA, SELECT ON SCHEMA datos_empresas.gold TO `grupo_recursos_humanos`;

-- Finanzas: solo lectura, solo sobre Gold
GRANT USE CATALOG ON CATALOG datos_empresas TO `grupo_finanzas`;
GRANT USE SCHEMA, SELECT ON SCHEMA datos_empresas.gold TO `grupo_finanzas`;

-- Auditar lo que quedo otorgado
SHOW GRANTS ON SCHEMA datos_empresas.gold;

**Estado real:** el grupo `admin` ya existe y esta cuenta ya es miembro -confirmado en vivo con `is_account_group_member('admin') = true`. El intento de correr el primer `GRANT` (`ALL PRIVILEGES ON CATALOG ... TO admin`) en esta sesion fue bloqueado automaticamente por el modo de permisos del asistente, por tratarse de una accion sensible (otorgar privilegios sobre el catalogo completo). Las 5 sentencias de esta celda quedan listas para correrlas manualmente desde un notebook de Databricks (o aprobando el permiso), una vez que `grupo_recursos_humanos` y `grupo_finanzas` tambien tengan a sus miembros agregados.

---
## Laboratorio 2: Enmascarado de columnas (Column Masks) y filtros de fila (Row Filters)

**Decision de diseno importante:** `numero_identificacion_normalizado` es la **llave de join** que usa `07_Gold_Fact_Contratos` para enlazar proveedores con el RUES. Si se aplicara un column mask directamente sobre `gold.de_dim_empresas`, cualquier identidad que no este en el grupo admin -incluido el propio pipeline, segun como este configurado el service principal- rompería ese join al ver `'***'` en vez del documento real.

Por eso, igual que en el laboratorio del profesor, las mascaras y filtros se practican sobre una **copia de solo lectura** en un esquema aislado (`governance_lab`), nunca sobre las tablas de produccion que usa el pipeline. Esto demuestra la tecnica con datos reales, sin arriesgar el funcionamiento del proyecto.

In [ ]:
%sql
-- Paso 1: esquema aislado para el laboratorio (no interfiere con silver/dql/gold de produccion)
CREATE SCHEMA IF NOT EXISTS datos_empresas.governance_lab
  COMMENT 'Esquema de practica para politicas de seguridad (TP3). No lo usa el pipeline productivo.';

-- Paso 2: copia real de dim_empresas para practicar sin riesgo sobre produccion
CREATE OR REPLACE TABLE datos_empresas.governance_lab.dim_empresas_demo
  CLUSTER BY (camara_comercio)
AS SELECT * FROM datos_empresas.gold.de_dim_empresas;

SELECT COUNT(*) AS filas_copiadas FROM datos_empresas.governance_lab.dim_empresas_demo;

**Resultado real:** 2.651.079 filas copiadas -exactamente las mismas que `gold.de_dim_empresas` en el momento de la copia. Se pueden aplicar mascaras y filtros sobre esta copia con la certeza de que ningun notebook de produccion la esta leyendo.

### Column masks: enmascarar el documento de identidad

In [ ]:
%sql
-- UDF de enmascarado: solo el grupo admin ve el documento real; RRHH, Finanzas y
-- cualquier otra identidad ven un valor parcial. (El grupo analistas de una version
-- anterior ya no aplica -ver nota del Laboratorio 1.)
CREATE OR REPLACE FUNCTION datos_empresas.governance_lab.mask_identificacion(val STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('admin') THEN val
  WHEN val IS NULL THEN NULL
  ELSE CONCAT(REPEAT('*', GREATEST(LENGTH(val) - 3, 0)), RIGHT(val, 3))
END;

-- UDF de enmascarado para razon_social (nombre de la empresa/persona natural)
CREATE OR REPLACE FUNCTION datos_empresas.governance_lab.mask_razon_social(val STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('admin') THEN val ELSE '*** RAZON SOCIAL OCULTA ***' END;

-- Aplicar las mascaras a la copia de practica
ALTER TABLE datos_empresas.governance_lab.dim_empresas_demo
  ALTER COLUMN numero_identificacion_normalizado SET MASK datos_empresas.governance_lab.mask_identificacion;

ALTER TABLE datos_empresas.governance_lab.dim_empresas_demo
  ALTER COLUMN nit_normalizado SET MASK datos_empresas.governance_lab.mask_identificacion;

ALTER TABLE datos_empresas.governance_lab.dim_empresas_demo
  ALTER COLUMN razon_social SET MASK datos_empresas.governance_lab.mask_razon_social;

-- Verificar: solo admin ve el documento y la razon social completos
SELECT llave_empresa, razon_social, numero_identificacion_normalizado, nit_normalizado, camara_comercio
FROM datos_empresas.governance_lab.dim_empresas_demo
LIMIT 10;

**Resultado real (ejecutado en vivo, actualizado):** mi cuenta ya es miembro del grupo real `admin` -lo cree en el Account Console y me agregue como parte del Laboratorio 2b-, asi que ahora veo `razon_social` y los documentos **completos**, sin enmascarar (ej. `DELGADO HENRY`, `79308614`). Antes de crear el grupo, con la misma consulta veia `*** RAZON SOCIAL OCULTA ***` y solo los ultimos 3 digitos del documento -ese es exactamente el resultado que deberia ver cualquier cuenta de `grupo_recursos_humanos` o `grupo_finanzas` (ninguno de los dos esta en `admin`). La mascara responde en tiempo real a la membresia del grupo: no hace falta tocar el SQL ni la tabla para que el resultado cambie.

### Row filter: limitar el acceso por camara de comercio

In [ ]:
%sql
-- Caso de uso real: un analista regional (grupo_recursos_humanos o grupo_finanzas) solo
-- deberia ver las empresas de su propia camara de comercio. Admin ve todas las camaras.
CREATE OR REPLACE FUNCTION datos_empresas.governance_lab.filter_por_camara(camara_val STRING)
RETURNS BOOLEAN
RETURN CASE WHEN is_account_group_member('admin') THEN TRUE
            ELSE camara_val = 'BOGOTA'
       END;

-- Nota: la sintaxis de la documentacion generica usa "USING COLUMNS (...)", pero en este
-- workspace (runtime actual) la forma que de verdad corre es "ON (...)". Se valido en vivo.
ALTER TABLE datos_empresas.governance_lab.dim_empresas_demo
  SET ROW FILTER datos_empresas.governance_lab.filter_por_camara ON (camara_comercio);

-- Verificar: solo deberian verse filas de BOGOTA si no sos admin
SELECT camara_comercio, COUNT(*) AS empresas
FROM datos_empresas.governance_lab.dim_empresas_demo
GROUP BY camara_comercio
ORDER BY empresas DESC;

**Resultado real (ejecutado en vivo, actualizado):** con mi cuenta ya en el grupo `admin`, el filtro me deja ver **las 58 camaras de comercio y las 2.651.079 filas completas** de la tabla copiada -el `CASE` de la funcion detecta `is_account_group_member('admin') = true` y devuelve `TRUE` sin aplicar el filtro. Antes de crear el grupo, la misma consulta devolvia solo **BOGOTA: 592.063 empresas** y ninguna otra camara -ese es el resultado que va a ver cualquier cuenta de `grupo_recursos_humanos` o `grupo_finanzas`. El filtro se aplica como un `WHERE` invisible y no se puede evitar ni con un `COUNT(*)` ni con un `SELECT *`.

---
## Laboratorio 2b: Enmascarado diferenciado por rol (ejercicio en vivo con compañeros)

Los dos laboratorios anteriores usaron una logica binaria: "admin ve todo, el resto ve lo mismo enmascarado". Este laboratorio es distinto -**cada rol pierde columnas diferentes**, sobre la misma tabla, demostrando que el enmascarado no es todo-o-nada:

| Rol | Quien | Ve `proveedor_adjudicado` (nombre empresa) | Ve `documento_proveedor_normalizado` | Ve `valor_contrato` / `valor_pagado` |
|---|---|---|---|---|
| **Admin** | Yo (owner del catalogo) | Si | Si | Si |
| **Recursos Humanos** | Companero 1 | Si | Si | **No** (NULL) |
| **Finanzas** | Companero 2 | Si | **No** (NULL) | Si |

La idea de negocio: RRHH necesita saber con que proveedor/persona se contrato (para gestion de terceros), pero no necesita ver cifras de dinero; Finanzas necesita las cifras para su analisis, pero no necesita el documento de identidad del proveedor (dato personal, minimizacion). El nombre de la empresa es el unico dato que ambos roles necesitan y por eso no se enmascara para nadie.

**Estado real del ejercicio:**

1. **Hecho:** grupo `admin` creado en el **Account Console** -> **User management** -> **Groups**, y esta cuenta agregada como miembro. Confirmado en vivo: `is_account_group_member('admin') = true`, y las mascaras/filtros de los Laboratorios 2 y 2b ya responden a eso (ver resultados actualizados arriba).
2. **Pendiente:** crear `grupo_recursos_humanos` y `grupo_finanzas`, agregar a **Companero 1** y **Companero 2** respectivamente (por su correo institucional), y darles `SELECT` sobre `datos_empresas.governance_lab.contratos_demo` (y `USE CATALOG`/`USE SCHEMA` sobre `datos_empresas`/`governance_lab`) -sin esto no van a poder ni conectarse a la tabla, independientemente de la mascara.
3. Cada quien corre su propio notebook de prueba desde su propia cuenta, en la carpeta `pruebas_roles/` de este mismo TP: `Prueba_Admin.ipynb`, `Prueba_RRHH.ipynb` y `Prueba_Finanzas.ipynb` -el resultado va a ser distinto para cada uno, sin cambiar ni una linea de SQL.

In [ ]:
%sql
-- Copia de practica: contratos con nombre del proveedor, documento, sector y valores.
-- Misma logica de aislamiento que dim_empresas_demo: nunca sobre la tabla de produccion.
CREATE OR REPLACE TABLE datos_empresas.governance_lab.contratos_demo
  CLUSTER BY (sector)
AS SELECT id_contrato, proveedor_adjudicado, documento_proveedor_normalizado,
          sector, fecha_firma, valor_contrato, valor_pagado
FROM datos_empresas.gold.de_fact_contratos;

SELECT COUNT(*) AS filas_copiadas FROM datos_empresas.governance_lab.contratos_demo;

**Resultado real:** 249.666 filas copiadas -las mismas que `gold.de_fact_contratos`.

### Mascaras: documento solo para admin/RRHH, valores solo para admin/Finanzas

In [ ]:
%sql
-- El documento del proveedor lo ven admin y RRHH; Finanzas lo ve NULL.
CREATE OR REPLACE FUNCTION datos_empresas.governance_lab.mask_documento_proveedor(val STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('admin') OR is_account_group_member('grupo_recursos_humanos') THEN val
  ELSE NULL
END;

-- Los valores del contrato los ven admin y Finanzas; RRHH los ve NULL.
CREATE OR REPLACE FUNCTION datos_empresas.governance_lab.mask_valor_contrato(val DECIMAL(22,2))
RETURNS DECIMAL(22,2)
RETURN CASE
  WHEN is_account_group_member('admin') OR is_account_group_member('grupo_finanzas') THEN val
  ELSE NULL
END;

-- Aplicar las mascaras (nombre de la empresa y sector quedan sin mascara: los ve cualquiera)
ALTER TABLE datos_empresas.governance_lab.contratos_demo
  ALTER COLUMN documento_proveedor_normalizado SET MASK datos_empresas.governance_lab.mask_documento_proveedor;

ALTER TABLE datos_empresas.governance_lab.contratos_demo
  ALTER COLUMN valor_contrato SET MASK datos_empresas.governance_lab.mask_valor_contrato;

ALTER TABLE datos_empresas.governance_lab.contratos_demo
  ALTER COLUMN valor_pagado SET MASK datos_empresas.governance_lab.mask_valor_contrato;

-- Verificacion: cada persona corre esta misma consulta desde su propia cuenta
SELECT id_contrato, proveedor_adjudicado, documento_proveedor_normalizado, sector, valor_contrato, valor_pagado
FROM datos_empresas.governance_lab.contratos_demo
LIMIT 5;

**Resultado real (ejecutado en vivo, actualizado):** mi cuenta ya es miembro del grupo `admin`, asi que la consulta devuelve **todas las columnas completas**: `documento_proveedor_normalizado`, `valor_contrato` y `valor_pagado` con su valor real (ej. `60349739`, `109.250.000`), ademas de `proveedor_adjudicado` y `sector`. Antes de crear el grupo `admin`, la misma consulta devolvia `NULL` en documento y en ambos valores -el escenario mas restrictivo, el mismo que va a ver cualquier cuenta que no este en ninguno de los 3 grupos.

**Para completar el ejercicio en vivo:** falta crear `grupo_recursos_humanos` y `grupo_finanzas` en el Account Console y agregar a los 2 companeros. Una vez hecho, cada uno corre `Prueba_RRHH.ipynb` o `Prueba_Finanzas.ipynb` (carpeta `pruebas_roles/`) desde su propia cuenta de Databricks. El companero en RRHH deberia ver el documento pero `NULL` en los valores; el de Finanzas, lo opuesto. Sin tocar el SQL, cada identidad real obtiene una respuesta distinta -la misma prueba que ya se confirmo con mi propia cuenta.

---
## Laboratorio 3: Auditoria de accesos

`system.access.audit` registra cada evento sobre Unity Catalog -lecturas, `GRANT`, `ALTER TABLE`, creacion de funciones- a nivel de toda la cuenta. Es el mecanismo que exige la Ley 1581 de 2012 (trazabilidad de quien accede a datos personales).

In [ ]:
%sql
-- Primer intento: system.access.audit (eventos de la API de Unity Catalog -grants, lecturas,
-- lineage-). Devolvio 0 filas para nuestras sentencias DDL: en este workspace, el DDL ejecutado
-- via el SQL Warehouse (Statement Execution API) no aparece ahi dentro de la ventana de latencia
-- normal del log. Lo que SI deja rastro completo y exacto es system.query.history: cada sentencia
-- SQL que corrimos, con su texto literal, quien la ejecuto y cuando.
SELECT start_time, statement_text, executed_by
FROM system.query.history
WHERE start_time >= DATEADD(HOUR, -4, CURRENT_TIMESTAMP())
  AND statement_text LIKE '%governance_lab%'
ORDER BY start_time DESC
LIMIT 15;

**Resultado real (ejecutado en vivo, dos intentos):**

1. `system.access.audit` filtrando por `action_name IN ('GRANT','REVOKE','ALTER','CREATE')` devolvio **0 filas**. Se investigo por que: en `system.access.audit`, `action_name` usa nombres tipo `createTable`, `alterTable`, `createFunction` (camelCase), no los verbos SQL en mayuscula -eso ya se corrigio al buscar- pero aun buscando por esos nombres camelCase en toda la cuenta (sin filtro de catalogo) en las ultimas 4 horas, **tampoco aparecio ninguno de nuestros `CREATE SCHEMA`/`CREATE TABLE`/`CREATE FUNCTION`/`ALTER TABLE`**. Lo que si quedo registrado ahi son llamadas de la API de metadatos de Unity Catalog (`listSchemas`, `getSchema`, `listTables`, `generateTemporaryTableCredential`) generadas por la navegacion en Data Explorer -es decir, `access.audit` en este workspace audita el plano de control de Unity Catalog, no necesariamente cada sentencia DDL ejecutada contra un SQL Warehouse vinculado via la Statement Execution API.
2. `system.query.history` si contiene el rastro completo: las **9 sentencias exactas** del Laboratorio 2 -desde `CREATE SCHEMA IF NOT EXISTS datos_empresas.governance_lab` (23:53:22) hasta la verificacion `SELECT ... LIMIT 10` (23:54:15)-, cada una con su texto literal completo y `executed_by = j.herrera2749@pascualbravo.edu.co`.

**Conclusion para el informe de gobernanza:** la trazabilidad de "quien cambio que y cuando" que exige la Ley 1581 de 2012 se puede resolver en este workspace con `system.query.history` (texto literal de cada sentencia) complementado por `system.access.audit` para los eventos de la API de Unity Catalog (grants, lecturas, gestion de metadatos). Ninguna de las dos tablas por si sola basta; se necesitan ambas segun el tipo de evento que se quiera auditar. Esto tambien es, en si mismo, un hallazgo de gobernanza: un equipo de datos nuevo en un workspace no deberia asumir que un solo system table captura todo -hay que validarlo con una prueba como esta antes de depender de el para cumplimiento.

---
## Resumen y buenas practicas aplicadas

| Pilar | Lo que se implemento | Evidencia real |
|---|---|---|
| RBAC | 3 grupos reales (`admin`, `grupo_recursos_humanos`, `grupo_finanzas`), `GRANT` por grupo (minimo privilegio) | `admin` ya creado y confirmado en vivo (`is_account_group_member = true`); `GRANT` real pendiente de aprobacion manual (bloqueado por el modo de permisos de esta sesion) |
| Column masks (dim_empresas_demo) | `numero_identificacion_normalizado`, `nit_normalizado` y `razon_social` solo los ve `admin`; cualquier otra identidad (RRHH, Finanzas, etc.) los ve enmascarados | Verificado en vivo, antes y despues de crear `admin`: enmascarado -> completo al agregarme al grupo |
| Column masks (por rol, Lab 2b) | Sobre `contratos_demo`: RRHH pierde `valor_contrato`/`valor_pagado` pero conserva el documento; Finanzas pierde el documento pero conserva los valores; el nombre de la empresa queda visible para ambos | Verificado en vivo: con `admin` veo todo completo; antes de crear el grupo veia ambas columnas en `NULL` -falta repetir con los 2 companeros reales |
| Row filter | Acceso limitado por `camara_comercio` para quien no es `admin` | Verificado en vivo, antes y despues de crear `admin`: 1 camara (BOGOTA, 592.063 filas) -> 58 camaras (2.651.079 filas completas) |
| Auditoria | Rastreo de cada sentencia DDL del laboratorio | Verificado en vivo via `system.query.history` (9/9 sentencias encontradas) |
| Minimizacion de datos | Nombres de personas naturales excluidos de Silver/Gold desde el diseno original | Ya aplicado desde `03_Silver_RUES` (TP2), no es nuevo en este notebook |

**Decision deliberada de alcance:** no se implemento ABAC (tags gobernados + politicas con `has_tag()`) porque el catalogo `datos_empresas` no tiene todavia una taxonomia de tags de sensibilidad definida, y el RBAC + column masks + row filters ya cubren los requisitos de la actividad para nuestro caso real. Se deja documentado como siguiente paso si el proyecto creciera a mas equipos con necesidades de acceso mas granulares que las de los 3 roles probados.